# MA: CAIO-M365 Instruction API Contract

**Purpose:** Document and verify the CAIO-M365 contract (MA Phases 2–4). This notebook ties the contract math to invariants and lemmas and runs the same checks as CI.

**Sources:**
- Intent: `docs/contracts/caio-m365/INTENT.md`
- Mathematics: `docs/contracts/caio-m365/MATHEMATICS.md` (Eq. 1–5, result shapes); Master: `M365_MASTER_CALCULUS.md`
- Action spec: `docs/contracts/caio-m365/ACTION_SPECIFICATION.md`
- Invariant: `invariants/INV-CAIO-M365-001.yaml`
- Lemmas: `docs/contracts/caio-m365/LEMMAS.md`, `LEMMA_L1.md`
- Verification script: `scripts/ci/verify_caio_m365_contract.py`
- CI: `.github/workflows/ci.yml` (MA contract verification step)

## Contract math (summary)

- **Eq. 1 (success):** \(\mathrm{ok}(S) = \mathrm{true} \Rightarrow \mathrm{result}(S) \neq \mathrm{null} \wedge \mathrm{shape}(\mathrm{result}(S)) \in \mathcal{S}_{\texttt{action}}\)
- **Eq. 2 (error):** \(\mathrm{ok}(S) = \mathrm{false} \Rightarrow \mathrm{error}(S) \neq \mathrm{null}\)

Implemented actions \(\mathcal{A}\): `list_users`, `list_teams`, `list_sites`, `get_user`, `reset_user_password`, `create_site`, `create_team`, `add_channel`, `provision_service`. Result shapes \(\mathcal{S}_{\texttt{action}}\) are defined in MATHEMATICS.md and ACTION_SPECIFICATION.md.

In [ ]:
# Run the contract verification script (same as CI).
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "scripts" / "ci" / "verify_caio_m365_contract.py").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
script = repo_root / "scripts" / "ci" / "verify_caio_m365_contract.py"
assert script.exists(), f"Not found: {script}"

result = subprocess.run([sys.executable, str(script)], cwd=str(repo_root), capture_output=True, text=True)
print(result.stdout or "")
if result.stderr:
    print(result.stderr, file=sys.stderr)
print("Exit code:", result.returncode)
assert result.returncode == 0, "Contract verification failed (CI would fail)"

In [ ]:
# Load and display the verification artifact (INV-CAIO-M365-001 acceptance).
import json

artifact_path = repo_root / "configs" / "generated" / "caio_m365_contract_verification.json"
with open(artifact_path) as f:
    artifact = json.load(f)

print("Postcondition pass:", artifact.get("postcondition_pass"))
print("Response schema pass:", artifact.get("response_schema_pass"))
print("Details (sample):", json.dumps({k: v for k, v in list(artifact.get("details", {}).items())[:5]}, indent=2))